# Lending Club Loan Performance & Risk Analysis (2007–2018)

Peer-to-peer lending platforms must carefully balance **credit risk** and **portfolio profitability**. If risk is mispriced or poorly managed, default rates rise, investor confidence falls, and the business becomes less sustainable.

This notebook uses historical Lending Club data (2007–2018) to understand **which borrowers and loans are more likely to default**, and **which segments are most profitable after losses**.

We will focus on questions such as:

- **How often do loans default overall?**
- **How does default risk vary by credit grade, income, loan size, and FICO score?**
- **Which segments generate the most profit after accounting for defaults?**
- **How has default risk evolved across vintages (issue years)?**

The analysis combines **SQL** (for business queries on the `loans` table) and **Power BI** (for interactive visualization) on top of a cleaned dataset stored in `cleaned_data.db`. Our goal is to produce **actionable insights** that support better underwriting, pricing, and portfolio management decisions.

---

## Dataset connection and overview

In this section we connect to the cleaned SQLite database (`cleaned_data.db`) and preview the `loans` table, which is the main fact table used throughout the analysis.

Some of the most important fields we will rely on are:

- **`grade`**: Lending Club credit grade assigned to the loan
- **`default`**: binary loan outcome (1 = defaulted, 0 = non-default)
- **`loan_amnt`**: original loan amount funded
- **`int_rate`**: interest rate charged on the loan
- **`annual_inc`**: borrower’s annual income
- **`dti`**: debt-to-income ratio at origination
- **`purpose`**: declared purpose for the loan (e.g., debt consolidation, business)
- **`fico_score`**: borrower’s FICO credit score
- **`issue_d`**: date the loan was issued

The query below simply pulls a small sample of rows so we can visually confirm the structure and values before computing any metrics.

In [3]:
%load_ext sql

%sql sqlite:///../data/cleaned_data.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [5]:
%%sql
SELECT *
FROM loans
LIMIT 5;

 * sqlite:///../data/cleaned_data.db
Done.


grade,delinq_2yrs,inq_last_6mths,dti,annual_inc,revol_util,loan_amnt,int_rate,term,issue_d,installment,earliest_cr_line,total_acc,open_acc,mths_since_last_delinq,total_pymnt,default,purpose_group,fico_score
C,0,1,5.91,55000.0,29.7,3600.0,13.99,36,2015-12-01 00:00:00.000000,123.03,2003-08-01 00:00:00.000000,13,7,30,4421.72,0,Debt,677.0
C,1,4,16.06,65000.0,19.2,24700.0,11.99,36,2015-12-01 00:00:00.000000,820.28,1999-12-01 00:00:00.000000,38,22,6,25679.66,0,Business,717.0
B,0,0,10.78,63000.0,56.2,20000.0,10.78,60,2015-12-01 00:00:00.000000,432.66,2000-08-01 00:00:00.000000,18,6,0,22705.92,0,Home,697.0
F,1,3,25.37,104433.0,64.5,10400.0,22.45,60,2015-12-01 00:00:00.000000,289.91,1998-06-01 00:00:00.000000,35,12,12,11740.5,0,Consumer,697.0
C,0,0,10.2,34000.0,68.4,11950.0,13.44,36,2015-12-01 00:00:00.000000,405.18,1987-10-01 00:00:00.000000,6,5,0,13708.95,0,Debt,692.0


---

## Default rates

In this section we quantify **how frequently loans default**, both at the portfolio level and across key borrower and loan segments. This is the foundation for understanding where risk is concentrated before layering on profitability.

We break the analysis into:

- **Total default rate**: overall share of loans that defaulted in the portfolio.
- **By credit grade**: how default risk changes from A to G.
- **By income level**: relationship between borrower income and default.
- **By loan amount**: how risk varies across small, medium, and large tickets.

### Total default rate

In [3]:
%%sql
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate_percent
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_loans,total_defaults,default_rate_percent
1371165,294415,21.47


**Insight:** Overall default rate is **21.47%** (294,415 defaults out of 1,371,165 loans). This is a meaningful credit-risk level, so segment-level monitoring (grade/FICO/DTI/loan size) and risk-based pricing are critical to protect portfolio performance.

### Default Rate by Credit Grade

In [4]:
%%sql
SELECT 
    grade,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,loans,defaults,default_rate
A,236758,15869,6.7
B,398528,58357,14.64
C,390724,94687,24.23
D,206696,66797,32.32
E,96224,38609,40.12
F,32828,15261,46.49
G,9407,4835,51.4


**Insight:** Default risk rises sharply as grade worsens (A: 6.7% → G: 51.4%). Grade is a strong risk discriminator and should be central to pricing, credit policy, and exposure limits.

### Default Rate by FICO Score

In [ ]:
%%sql
SELECT 
    CASE
        WHEN fico_score < 600 THEN 'Poor'
        WHEN fico_score < 700 THEN 'Fair'
        WHEN fico_score < 750 THEN 'Good'
        ELSE 'Excellent'
    END AS credit_group,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate

FROM loans
GROUP BY credit_group;

 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate
Excellent,108092,10964,10.14
Fair,837712,210540,25.13
Good,425361,72911,17.14


**Insight:** Default rates are lowest for **Excellent** credit and highest for **Fair** borrowers, which also make up the largest share of the book. This means a relatively small deterioration in credit quality (from Good to Fair) leads to a disproportionately higher default risk and should be a key focus for underwriting and exposure limits.

### Default Rate by Income Level

In [5]:
%%sql
SELECT 
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate
    
FROM loans
GROUP BY income_group;

 * sqlite:///../data/cleaned_data.db
Done.


income_group,loans,defaults,default_rate
High Income,480682,88863,18.49
Low Income,214632,54213,25.26
Middle Income,675851,151339,22.39


**Insight:** Lower-income borrowers show higher default rates, but the gap is smaller than for grade/FICO. Income is informative, but it’s most useful when combined with affordability metrics (e.g., DTI) and credit quality.

### Default Rate by Loan Amount

In [6]:
%%sql
SELECT 
    CASE
        WHEN loan_amnt < 5000 THEN 'Small'
        WHEN loan_amnt < 15000 THEN 'Medium'
        ELSE 'Large'
    END AS loan_size,
    
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate

FROM loans
GROUP BY loan_size;

 * sqlite:///../data/cleaned_data.db
Done.


loan_size,loans,defaults,default_rate
Large,590009,144248,24.45
Medium,646277,127803,19.78
Small,134879,22364,16.58


**Insight:** Larger loans have higher default rates (Large: 24.45% vs Small: 16.58%), suggesting both higher borrower risk and higher exposure. Consider tighter underwriting or pricing adjustments for larger balances.

## Interest

### Average Interest Rate by Grade

In [7]:
%%sql
SELECT 
    grade,
    ROUND(AVG(int_rate),2) AS avg_interest_rate
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,avg_interest_rate
A,7.11
B,10.68
C,14.03
D,17.75
E,21.21
F,25.0
G,27.8


### Total Net Payments Minus Principal

In [8]:
%%sql
SELECT 
    ROUND(SUM(total_pymnt - loan_amnt),2) AS total_profit
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_profit
337352631.21


### Net Payments Minus Principal by Grade

In [9]:
%%sql
SELECT 
    grade,
    ROUND(SUM(total_pymnt - loan_amnt),2) AS profit
FROM loans
GROUP BY grade
ORDER BY profit DESC;

 * sqlite:///../data/cleaned_data.db
Done.


grade,profit
B,223398529.7
A,159560656.3
C,73171868.86
G,-18312837.6
F,-29219974.8
D,-32053846.67
E,-39191764.58


 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate
Excellent,108092,10964,10.14
Fair,837712,210540,25.13
Good,425361,72911,17.14


---

## Additional risk views

### Exposure-weighted default rate (by dollars)
This measures the share of **loan amount** that defaulted (not just the share of loans), which is often more relevant for loss exposure.

In [ ]:
%%sql
SELECT
  ROUND(
    100.0 * SUM(CASE WHEN "default" = 1 THEN loan_amnt ELSE 0 END) / NULLIF(SUM(loan_amnt), 0),
    2
  ) AS exposure_weighted_default_rate_percent,
  SUM(loan_amnt) AS total_loan_amount,
  SUM(CASE WHEN "default" = 1 THEN loan_amnt ELSE 0 END) AS defaulted_loan_amount
FROM loans;

### Default rate trend by issue year (vintage)
This helps identify whether defaults are stable over time or driven by specific lending vintages / macro conditions.

In [ ]:
%%sql
SELECT
  strftime('%Y', issue_d) AS issue_year,
  COUNT(*) AS loans,
  SUM("default") AS defaults,
  ROUND(100.0 * SUM("default") / NULLIF(COUNT(*), 0), 2) AS default_rate_percent
FROM loans
GROUP BY issue_year
ORDER BY issue_year;